# Sentiment Analysis Pipeline
Make sure you enable the GPU in Google Colab (Runtime > Change runtime type > Hardware accelerator > T4 GPU).

### Block 1: Install Required Libraries

In [ ]:
!pip install transformers datasets accelerate scikit-learn pandas matplotlib seaborn

### Block 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

# Hugging Face imports
import torch
from transformers import BartTokenizer, BartForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

import warnings
warnings.filterwarnings('ignore')

# Dictionary to store all F1 scores for final comparison
f1_scores = {}

### Block 3: Load Data & Train/Test Split
*Note: Upload your CSV file to Colab first and replace `'your_dataset.csv'` with the actual file name.*

In [ ]:
# Load your dataset
# Replace with your actual file path
df = pd.read_csv('your_dataset.csv')

# Drop any rows with missing text or labels
df = df.dropna(subset=['processed_text', 'sentiment_numeric'])

# Map target variable so it starts from 0 (required for Neural Networks like BART)
unique_labels = sorted(df['sentiment_numeric'].unique())
label_mapping = {val: idx for idx, val in enumerate(unique_labels)}
df['sentiment_mapped'] = df['sentiment_numeric'].map(label_mapping)

# Features and Labels
X = df['processed_text'].astype(str)
y = df['sentiment_mapped'].astype(int)

# Train-test split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print(f"Number of distinct sentiment classes: {len(unique_labels)}")

### Block 4: Feature Extraction (TF-IDF)

In [ ]:
# Initialize TF-IDF Vectorizer (limiting to 5000 features for memory efficiency)
tfidf = TfidfVectorizer(max_features=5000)

# Fit on training data and transform both train and test data
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF feature matrix shape: {X_train_tfidf.shape}")

### Block 5: Naive Bayes Model

In [ ]:
print("--- Training Naive Bayes ---")
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

# Predict and evaluate
nb_preds = nb_model.predict(X_test_tfidf)
nb_f1 = f1_score(y_test, nb_preds, average='weighted')
f1_scores['Naive Bayes'] = nb_f1

print(f"Naive Bayes F1 Score: {nb_f1:.4f}")

# Plot Confusion Matrix
cm = confusion_matrix(y_test, nb_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title('Naive Bayes Confusion Matrix')
plt.show()

### Block 6: Logistic Regression Model

In [ ]:
print("--- Training Logistic Regression ---")
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

# Predict and evaluate
lr_preds = lr_model.predict(X_test_tfidf)
lr_f1 = f1_score(y_test, lr_preds, average='weighted')
f1_scores['Logistic Regression'] = lr_f1

print(f"Logistic Regression F1 Score: {lr_f1:.4f}")

# Plot Confusion Matrix
cm = confusion_matrix(y_test, lr_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title('Logistic Regression Confusion Matrix')
plt.show()

### Block 7: Random Forest Model

In [ ]:
print("--- Training Random Forest ---")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_tfidf, y_train)

# Predict and evaluate
rf_preds = rf_model.predict(X_test_tfidf)
rf_f1 = f1_score(y_test, rf_preds, average='weighted')
f1_scores['Random Forest'] = rf_f1

print(f"Random Forest F1 Score: {rf_f1:.4f}")

# Plot Confusion Matrix
cm = confusion_matrix(y_test, rf_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title('Random Forest Confusion Matrix')
plt.show()

### Block 8: Preparing BART Model

In [ ]:
print("--- Preparing Data for BART ---")
num_labels = len(unique_labels)
model_name = "facebook/bart-base"

# Load Tokenizer
tokenizer = BartTokenizer.from_pretrained(model_name)

# Create Hugging Face Datasets
train_df = pd.DataFrame({'text': X_train, 'label': y_train})
test_df = pd.DataFrame({'text': X_test, 'label': y_test})

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenization function
# max_length REDUCED to 64 for much faster processing
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

### Block 9: Training & Evaluating BART Model

In [ ]:
print("--- Training BART ---")
bart_model = BartForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# Define custom metrics function to calculate F1 during training
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits[0] if isinstance(logits, tuple) else logits, axis=-1)
    f1 = f1_score(labels, predictions, average='weighted')
    return {"f1": f1}

training_args = TrainingArguments(
    output_dir="./bart_results",
    eval_strategy="no", # Disabled evaluation during training for speed
    learning_rate=2e-5,
    per_device_train_batch_size=32, # Increased batch size
    per_device_eval_batch_size=32,
    num_train_epochs=1, # 1 Epoch
    max_steps=20, # Ultra-fast: stop training after only 20 steps
    weight_decay=0.01,
    fp16=True, # Enable Mixed Precision (FP16) to speed up GPU processing significantly
)

trainer = Trainer(
    model=bart_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

# Get predictions for test set
bart_preds_output = trainer.predict(tokenized_test)
bart_preds = np.argmax(bart_preds_output.predictions[0] if isinstance(bart_preds_output.predictions, tuple) else bart_preds_output.predictions, axis=-1)

# Calculate F1
bart_f1 = f1_score(y_test, bart_preds, average='weighted')
f1_scores['BART'] = bart_f1
print(f"\nBART F1 Score: {bart_f1:.4f}")

# Plot Confusion Matrix
cm_bart = confusion_matrix(y_test, bart_preds)
disp_bart = ConfusionMatrixDisplay(confusion_matrix=cm_bart)
disp_bart.plot(cmap='Blues')
plt.title('BART Confusion Matrix')
plt.show()

### Block 10: Model Comparison and Final Decision

In [ ]:
print("--- Final Model Comparison ---")

# Plotting F1 Scores
plt.figure(figsize=(10, 6))
sns.barplot(x=list(f1_scores.keys()), y=list(f1_scores.values()), palette='viridis')
plt.title('F1 Score Comparison of All Models')
plt.ylabel('Weighted F1 Score')
plt.xlabel('Models')
plt.ylim(0, 1)

# Add text labels on top of the bars
for i, v in enumerate(f1_scores.values()):
    plt.text(i, v + 0.01, f"{v:.4f}", ha='center', fontweight='bold')
plt.show()

# Find the best model
best_model_name = max(f1_scores, key=f1_scores.get)
best_model_score = f1_scores[best_model_name]

print("\n" + "="*40)
print("             CONCLUSION")
print("="*40)
print(f"The best performing model is: >> {best_model_name} <<")
print(f"With an F1 Score of: {best_model_score:.4f}")
print("="*40)